# 03.3 — Training at Scale and Debugging

The tools that make big training runs fit, run fast and stay stable: gradient clipping, mixed precision, gradient accumulation,
activation checkpointing, experiment tracking with MLflow, and multi-GPU training with DDP and FSDP. It ends with how to debug a run that isn't learning.

## What's in this notebook

- 1\. What changes when training gets big
- 2\. Gradient clipping
- 3\. Mixed precision
- 4\. Gradient accumulation
- 5\. Activation checkpointing
- 6\. Experiment tracking with MLflow
- 7\. Multi-GPU training: DDP, FSDP and beyond
- 8\. Debugging a training run
- 9\. Trade-offs
- 10\. Interview questions
- 11\. In my own words

In [1]:
import gzip
import hashlib
import json
import logging
import math
import os
import subprocess
import sys
import time
import urllib.request
import uuid
import warnings
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow  # noqa: E402
import numpy as np  # noqa: E402
import torch  # noqa: E402
from torch import nn  # noqa: E402
from torch.utils.checkpoint import checkpoint  # noqa: E402

DATA_DIR = Path("../data/fashion-mnist")
URL = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/"
MD5 = {"train-images-idx3-ubyte.gz": "8d4fb7e6c68d591d4c3dfef9ec88bf0d", "train-labels-idx1-ubyte.gz": "25c81989df183df01b3e8a0aad5dffbe"}


def load(name: str, offset: int) -> torch.Tensor:
    """Download a FashionMNIST file once, check its MD5, and return its values (see 02.2)."""
    path = DATA_DIR / name
    if not path.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(URL + name, path)
    assert hashlib.md5(path.read_bytes()).hexdigest() == MD5[name], f"{name}: bad checksum"
    with gzip.open(path) as f:
        return torch.from_numpy(np.frombuffer(f.read(), np.uint8, offset=offset).copy())


X = load("train-images-idx3-ubyte.gz", 16).reshape(-1, 784).float() / 255
y = load("train-labels-idx1-ubyte.gz", 8).long()
X = (X - X[:54_000].mean()) / X[:54_000].std()
X_train, y_train, X_val, y_val = X[:54_000], y[:54_000], X[54_000:], y[54_000:]
BATCH = 128


def make_mlp() -> nn.Module:
    return nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))


def accuracy(model: nn.Module) -> float:
    with torch.no_grad():
        return (model(X_val).argmax(1) == y_val).float().mean().item()


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def synchronize(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def show_error(fn) -> None:
    try:
        fn()
        print("no error")
    except Exception as e:
        print(f"{type(e).__name__}: {str(e).splitlines()[0]}")


device = get_device()
print("device:", device)

device: mps


# 1. WHAT CHANGES WHEN TRAINING GETS BIG

| Problem | Tool | Section |
|---|---|---|
| a few huge gradients wreck training | gradient clipping | 2 |
| training is slow and activations fill GPU memory | mixed precision (bfloat16 / float16) | 3 |
| the batch you want doesn't fit in memory | gradient accumulation | 4 |
| activations of a deep model don't fit | activation (gradient) checkpointing | 5 |
| dozens of runs, nobody remembers which settings gave what | experiment tracking (MLflow, W&B) | 6 |
| one GPU is too slow, or the model doesn't fit on one | DDP, FSDP / ZeRO, tensor and pipeline parallelism | 7 |
| Python overhead and many small GPU operations | `torch.compile` (folder 14) | — |
| the run isn't learning and you don't know why | a debugging checklist | 8 |

# 2. GRADIENT CLIPPING

Sometimes one batch produces a huge gradient (a rare example, a high learning rate, an RNN through a long sequence), and one bad step can undo hours of training or turn the weights into NaN.

**Clipping by norm** treats all gradients as one long vector; if its length (L2 norm) is above `max_norm`, the whole vector is scaled down to `max_norm`.
The direction is kept and only the size is capped. (Clipping each value separately, `clip_grad_value_`, changes the direction and is rarer.)

- Call it after `loss.backward()` and before `optimizer.step()`.
- `clip_grad_norm_` returns the norm *before* clipping. Log it: spikes in the gradient norm are an early warning of trouble.
- `max_norm = 1.0` is the usual value for transformers and LLMs.

In [2]:
def train_clip(lr: float, max_norm: float | None, epochs: int = 2) -> tuple[float, float, float]:
    torch.manual_seed(0)
    model = make_mlp()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    order = torch.Generator().manual_seed(0)
    norms = []
    for _ in range(epochs):
        perm = torch.randperm(len(X_train), generator=order)
        for i in range(len(X_train) // BATCH):
            idx = perm[i * BATCH:(i + 1) * BATCH]
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(X_train[idx]), y_train[idx])
            loss.backward()
            norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm if max_norm else float("inf"))
            norms.append(norm.item())
            optimizer.step()
    return accuracy(model), max(norms), float(np.median(norms))


for lr in [0.2, 0.5]:
    for max_norm in [None, 1.0]:
        acc, biggest, typical = train_clip(lr, max_norm)
        print(f"lr {lr}, {'no clipping ' if max_norm is None else 'clip at 1.0 '}: validation accuracy {acc:.4f}   "
              f"gradient norm: typical {typical:.2f}, largest {biggest:,.1f}")

lr 0.2, no clipping : validation accuracy 0.8163   gradient norm: typical 0.89, largest 6.1


lr 0.2, clip at 1.0 : validation accuracy 0.8397   gradient norm: typical 0.91, largest 9.4


lr 0.5, no clipping : validation accuracy 0.1032   gradient norm: typical 0.09, largest 387,588.9


lr 0.5, clip at 1.0 : validation accuracy 0.4445   gradient norm: typical 1.56, largest 38.7


- **lr 0.2:** the typical gradient norm is under 1, but the biggest spikes are 6–9 times larger. Clipping them at 1.0 raises validation accuracy from 0.816 to 0.840.
- **lr 0.5, no clipping:** the run blows up. One gradient reaches a norm of 387,589, the weights are ruined, and accuracy falls to chance (0.103). With clipping the same run survives (0.445).

Clipping is a safety net, not a cure for a learning rate that is too high: lr 0.5 is still far worse than 0.2.

# 3. MIXED PRECISION

**Mixed precision** runs most of the maths in 16-bit floats while keeping the parts that need precision in float32 (02.1 section 3 compared the formats).
- `torch.autocast` picks the precision per operation: matrix multiplies and convolutions run in 16 bits; softmax, normalization, reductions and losses stay in float32.
- The weights themselves stay in float32 ("master weights"), so small updates are not lost.
- Gains: on NVIDIA GPUs, Tensor Cores make 16-bit matrix maths several times faster, and activations take half the memory.
- **bfloat16** has the float32 range, so it needs nothing else. It needs a recent GPU (A100, H100, recent consumer cards) or a TPU.
- **float16** has a tiny range: gradients below about 6e-8 become exactly zero. **Loss scaling** fixes this: multiply the loss by a large number (for example 65,536)
  before `backward()`, so the gradients are scaled up too, then divide them back in float32 before the update. `GradScaler` does this automatically
  and lowers the scale whenever an overflow (inf) appears.
- H100-class GPUs also support 8-bit floats (FP8) for even faster LLM training.

In [3]:
tiny = torch.tensor(1e-8)                              # a realistic small gradient
scale = 65536.0
print(f"float16 of 1e-8:                       {tiny.half().item()}")
print(f"float16 of 1e-8 x 65,536:              {(tiny * scale).half().item():.3e}")
print(f"... converted back and divided in fp32: {(tiny * scale).half().float().item() / scale:.3e}")

float16 of 1e-8:                       0.0
float16 of 1e-8 x 65,536:              6.552e-04
... converted back and divided in fp32: 9.997e-09


Without scaling, the gradient becomes exactly 0 in float16 and that weight stops learning. Scaled, it survives as 6.6e-4 and comes back as 1.0e-8.

Speed on this machine's GPU: a larger network (8 layers of 2,048) with a batch of 512, in float32 and with autocast.

In [4]:
def big_mlp() -> nn.Module:
    return nn.Sequential(*[m for _ in range(8) for m in (nn.Linear(2048, 2048), nn.ReLU())]).to(device)


xb = torch.randn(512, 2048, device=device)
for dtype in [None, torch.float16, torch.bfloat16]:
    torch.manual_seed(0)
    model = big_mlp()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

    def step():
        optimizer.zero_grad()
        with torch.autocast(device.type, dtype=dtype, enabled=dtype is not None):
            loss = model(xb).float().pow(2).mean()
        loss.backward()
        optimizer.step()
        return loss

    for _ in range(3):                                    # warm-up
        step()
    synchronize(device)
    start = time.perf_counter()
    for _ in range(20):
        loss = step()
    synchronize(device)
    print(f"{'float32' if dtype is None else 'autocast ' + str(dtype).split('.')[1]:<18} {(time.perf_counter() - start) / 20 * 1000:6.1f} ms per step   loss {loss.item():.6f}")

float32              32.3 ms per step   loss 0.000100


autocast float16     26.6 ms per step   loss 0.000100


autocast bfloat16    24.3 ms per step   loss 0.000100


On this Mac's GPU, autocast is 1.2–1.3x faster: 24 ms per step with bfloat16 and 27 ms with float16, against 32 ms in float32. The loss is the same to 6 decimal places. On NVIDIA GPUs with Tensor Cores the speed-up is usually larger, often 2x or more.

The standard float16 training step with `GradScaler`. With bfloat16, drop the scaler and keep only the `autocast` block.

In [5]:
torch.manual_seed(0)
model = make_mlp().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler(device.type)
Xd, yd = X_train[:BATCH * 50].to(device), y_train[:BATCH * 50].to(device)

for i in range(50):
    xb, yb = Xd[i * BATCH:(i + 1) * BATCH], yd[i * BATCH:(i + 1) * BATCH]
    optimizer.zero_grad()
    with torch.autocast(device.type, dtype=torch.float16):
        loss = nn.functional.cross_entropy(model(xb), yb)
    scaler.scale(loss).backward()                          # backward on the scaled loss
    scaler.unscale_(optimizer)                             # divide the gradients back before clipping
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)                                 # skips the step if any gradient is inf or NaN
    scaler.update()                                        # adjusts the scale for the next step
print(f"after 50 float16 steps: loss {loss.item():.4f}, loss scale {scaler.get_scale():,.0f}")

after 50 float16 steps: loss 0.6280, loss scale 65,536


# 4. GRADIENT ACCUMULATION

When the batch you want (say 128) doesn't fit in memory, run several smaller **micro-batches** (4 × 32), call `backward()` on each without `zero_grad()`,
and step the optimizer once. Gradients add up (02.1 section 9), so the update is the same as for the full batch, at the cost of more time per step.
- Divide each micro-batch loss by the number of micro-batches, so the sum equals the mean over the whole batch.
- BatchNorm still only sees 32 examples at a time, so its statistics are noisier. LayerNorm models (transformers) are unaffected.
- LLM training combines it with data parallelism: effective batch = micro-batch × accumulation steps × number of GPUs.

In [6]:
torch.manual_seed(0)
model = make_mlp()
xb, yb = X_train[:128], y_train[:128]

model.zero_grad()
nn.functional.cross_entropy(model(xb), yb).backward()
full_batch = [p.grad.clone() for p in model.parameters()]

model.zero_grad()
ACCUMULATION = 4
for chunk in range(ACCUMULATION):
    part = slice(chunk * 32, (chunk + 1) * 32)
    (nn.functional.cross_entropy(model(xb[part]), yb[part]) / ACCUMULATION).backward()     # gradients add up
accumulated = [p.grad.clone() for p in model.parameters()]

print("largest difference, one batch of 128 vs 4 accumulated micro-batches of 32:",
      f"{max((a - b).abs().max().item() for a, b in zip(full_batch, accumulated)):.1e}")

largest difference, one batch of 128 vs 4 accumulated micro-batches of 32: 7.5e-09


The gradients match to floating-point rounding (7.5e-9), so accumulation gives the same update as the bigger batch.

# 5. ACTIVATION CHECKPOINTING

The forward pass saves every layer's output for the backward pass, and for deep models these **activations** use more memory than the weights.
**Activation checkpointing** (also called gradient checkpointing) saves only the inputs of chosen blocks and recomputes the rest during the backward pass:
less memory, roughly one extra forward pass of compute. Hugging Face models switch it on with `model.gradient_checkpointing_enable()`, and it's standard when fine-tuning LLMs.

In [7]:
class DeepNet(nn.Module):
    def __init__(self, use_checkpoint: bool):
        super().__init__()
        self.blocks = nn.ModuleList([nn.Sequential(nn.Linear(1024, 1024), nn.GELU()) for _ in range(24)])
        self.use_checkpoint = use_checkpoint

    def forward(self, x):
        for block in self.blocks:
            x = checkpoint(block, x, use_reentrant=False) if self.use_checkpoint else block(x)
        return x


def allocated() -> int:
    return torch.cuda.memory_allocated() if device.type == "cuda" else torch.mps.current_allocated_memory() if device.type == "mps" else 0


x = torch.randn(4096, 1024, device=device)
for use_checkpoint in [False, True]:
    model = DeepNet(use_checkpoint).to(device)
    synchronize(device)
    before = allocated()
    out = model(x).sum()
    synchronize(device)
    activations_mb = (allocated() - before) / 1e6
    start = time.perf_counter()
    out.backward()
    synchronize(device)
    print(f"checkpointing {'on ' if use_checkpoint else 'off'}: memory held for backward {activations_mb:6.0f} MB, backward pass {(time.perf_counter() - start) * 1000:.0f} ms")
    del model, out

checkpointing off: memory held for backward    789 MB, backward pass 98 ms


checkpointing on : memory held for backward    386 MB, backward pass 156 ms


With checkpointing, the memory kept for the backward pass halves (789 MB → 386 MB), and the backward pass takes longer (98 ms → 156 ms) because each block's forward pass runs again. Here only each block's input is kept; transformer layers create many more intermediate tensors per block, so the saving there is larger.

# 6. EXPERIMENT TRACKING WITH MLFLOW

After a few dozen runs, nobody remembers which settings produced which result. An **experiment tracker** records every run's parameters, metrics per epoch,
and files (plots, configs, models) in one searchable place.
- **MLflow:** open source and self-hosted; also has a model registry and LLM tracing. Here it stores everything in a local SQLite file under `mlruns/` (not in git).
- **Weights & Biases:** hosted service with a strong dashboard; free for personal use, needs an account.
- To browse the runs: `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db`, run from the repository root, then open the printed address.

In [8]:
for name in ["mlflow", "alembic"]:                          # keep MLflow's setup messages quiet
    logging.getLogger(name).setLevel(logging.ERROR)
MLRUNS = Path("../mlruns").resolve()
MLRUNS.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{MLRUNS / 'mlflow.db'}")
EXPERIMENT = "fashion-mnist-learning-rate"
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=(MLRUNS / "artifacts").as_uri())
mlflow.set_experiment(EXPERIMENT)
session = uuid.uuid4().hex[:8]                              # tags the runs from this execution of the notebook

for lr in [1e-4, 1e-3, 1e-2]:
    with mlflow.start_run(run_name=f"adam-lr-{lr:g}"):
        mlflow.set_tag("session", session)
        mlflow.log_params({"optimizer": "Adam", "lr": lr, "batch_size": BATCH, "hidden": "256,128"})
        torch.manual_seed(0)
        model = make_mlp()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        order = torch.Generator().manual_seed(0)
        history = []
        for epoch in range(3):
            perm = torch.randperm(len(X_train), generator=order)
            for i in range(len(X_train) // BATCH):
                idx = perm[i * BATCH:(i + 1) * BATCH]
                optimizer.zero_grad()
                loss = nn.functional.cross_entropy(model(X_train[idx]), y_train[idx])
                loss.backward()
                optimizer.step()
            val_acc = accuracy(model)
            mlflow.log_metrics({"train_loss": loss.item(), "val_acc": val_acc}, step=epoch + 1)
            history.append({"epoch": epoch + 1, "val_acc": val_acc})
        mlflow.log_dict({"history": history}, "history.json")       # a file stored with the run

runs = mlflow.search_runs(filter_string=f"tags.session = '{session}'", order_by=["metrics.val_acc DESC"])
print(runs[["tags.mlflow.runName", "params.lr", "metrics.val_acc", "metrics.train_loss"]].to_string(index=False))

tags.mlflow.runName params.lr  metrics.val_acc  metrics.train_loss
      adam-lr-0.001     0.001         0.875500            0.256570
     adam-lr-0.0001    0.0001         0.846667            0.359686
       adam-lr-0.01      0.01         0.841833            0.401486


All three runs are stored with their settings, per-epoch metrics and a history file. Sorted by validation accuracy, lr 1e-3 wins (0.876), ahead of 1e-4 (0.847, too slow in 3 epochs) and 1e-2 (0.842, too noisy). The same query works across hundreds of runs, and the MLflow UI plots their curves side by side.

# 7. MULTI-GPU TRAINING: DDP, FSDP AND BEYOND

| Strategy | What is split across GPUs | When to use it |
|---|---|---|
| **DDP** (DistributedDataParallel) | the data: every GPU holds a full model copy and trains on a different slice of each batch; gradients are averaged (all-reduce) after `backward()` | the model fits on one GPU; you want more speed. The default |
| **FSDP** / **ZeRO** (DeepSpeed) | also the weights, gradients and optimizer state, gathered layer by layer when needed | the model plus Adam state doesn't fit on one GPU (LLM training and full fine-tuning) |
| **tensor parallelism** | each weight matrix, split across GPUs | single layers too big for one GPU; very large LLMs |
| **pipeline parallelism** | groups of layers, one group per GPU, with micro-batches flowing through | very deep models across many GPUs |

Large LLMs combine all of them ("3D parallelism"). Key DDP facts:
- **torchrun** starts one process per GPU and sets `RANK`, `LOCAL_RANK` and `WORLD_SIZE` for each.
- **DistributedSampler** gives each process a different slice of the data; call `sampler.set_epoch(epoch)` so the shuffle changes every epoch.
- **Effective batch** = batch per GPU × number of GPUs (× accumulation steps); scale the learning rate to match (03.1 section 11).
- Only rank 0 logs and saves checkpoints. The backend is `nccl` on NVIDIA GPUs and `gloo` on CPUs.

[`ddp_train.py`](ddp_train.py) is a complete DDP script. It runs here as 2 processes on the CPU with `gloo`; on a GPU server the same code runs with `nccl`.

In [9]:
print(Path("ddp_train.py").read_text().split('"""', 2)[2].strip())

import os

import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset


def main() -> None:
    use_cuda = torch.cuda.is_available()
    dist.init_process_group(backend="nccl" if use_cuda else "gloo")    # reads RANK and WORLD_SIZE set by torchrun
    rank, world_size = dist.get_rank(), dist.get_world_size()
    device = torch.device(f"cuda:{int(os.environ['LOCAL_RANK'])}") if use_cuda else torch.device("cpu")

    torch.manual_seed(0)                                   # same data and same starting weights in every process
    X = torch.randn(4096, 20)
    y = (X[:, :5].sum(dim=1) > 0).long()
    dataset = TensorDataset(X, y)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True, seed=0)
    loader = DataLoader(dataset, batch_size=64, sampler=sampler)

    model = DDP(nn.Sequential(nn.Linear(20, 64), nn.ReLU

In [10]:
env = {**os.environ, "OMP_NUM_THREADS": "1", "MASTER_ADDR": "127.0.0.1"}
if sys.platform == "darwin":
    env["GLOO_SOCKET_IFNAME"] = "lo0"                      # use the loopback network interface on macOS
command = [sys.executable, "-m", "torch.distributed.run",  # the same as the `torchrun` command
           "--nnodes", "1", "--nproc_per_node", "2", "--node-rank", "0",
           "--rdzv-backend", "static", "--master-addr", "127.0.0.1", "--master-port", "29511", "ddp_train.py"]
result = subprocess.run(command, capture_output=True, text=True, env=env, timeout=300)
print(result.stdout if result.returncode == 0 else result.stderr[-2000:])

epoch 0: each process trained on 2048 of 4096 examples, mean last-batch loss 0.4700
epoch 1: each process trained on 2048 of 4096 examples, mean last-batch loss 0.2792
epoch 2: each process trained on 2048 of 4096 examples, mean last-batch loss 0.2337
sum of all weights in each process: [2.734985, 2.734985] -> identical copies



Each process trained on 2,048 of the 4,096 examples per epoch (its half, from DistributedSampler). After 3 epochs, the sum of all weights is identical in both processes: averaging the gradients kept the two copies in sync. torchrun also writes some setup warnings to stderr; the cell shows them only if the run fails.

# 8. DEBUGGING A TRAINING RUN

A model that "doesn't learn" is usually a bug, not a modelling problem. Work through this list in order:

| Check | What it catches |
|---|---|
| look at a few examples and labels after the DataLoader | wrong labels, misaligned shuffling, broken preprocessing |
| loss at initialization ≈ −ln(1/classes) | wrong loss, a softmax applied twice, bad initialization |
| overfit one small batch to near-zero loss (02.3) | broken model, loss, optimizer or data pipeline |
| log the gradient norm (section 2) | exploding or vanishing gradients, dead layers |
| try a few learning rates on a log scale (03.1) | a learning rate off by 10x |
| compare with a simple baseline (logistic regression, a known good config) | an expected score that is wrong in the first place |
| turn on anomaly detection when NaNs appear | the exact operation that produced the first NaN |
| check `model.train()` / `model.eval()` and `zero_grad()` | the silent PyTorch bugs from 02.1 and 02.2 |

Three of these checks, in code. First, the loss at initialization and a data bug that keeps a model from learning anything.

In [11]:
torch.manual_seed(0)
model = make_mlp()
with torch.no_grad():
    init_loss = nn.functional.cross_entropy(model(X_train[:1000]), y_train[:1000]).item()
print(f"loss at initialization {init_loss:.3f}; expected about -ln(1/10) = {math.log(10):.3f}")


def one_epoch(labels: torch.Tensor) -> float:
    torch.manual_seed(0)
    model = make_mlp()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for i in range(len(X_train) // BATCH):
        idx = slice(i * BATCH, (i + 1) * BATCH)
        optimizer.zero_grad()
        nn.functional.cross_entropy(model(X_train[idx]), labels[idx]).backward()
        optimizer.step()
    return accuracy(model)


misaligned = y_train[torch.randperm(len(y_train), generator=torch.Generator().manual_seed(0))]   # images and labels shuffled separately
print(f"one epoch, correct labels: validation accuracy {one_epoch(y_train):.3f}")
print(f"one epoch, labels shuffled separately from the images: {one_epoch(misaligned):.3f}")

loss at initialization 2.327; expected about -ln(1/10) = 2.303


one epoch, correct labels: validation accuracy 0.861


one epoch, labels shuffled separately from the images: 0.090


The loss at initialization is 2.327, close to the expected 2.303, so the model and loss are wired correctly. With labels shuffled separately from the images, training runs without any error but stays at chance (0.090 against 0.861). Only looking at the data, or comparing with a baseline, catches this kind of bug.

Second, finding where a NaN comes from. `sqrt` has an infinite slope at 0, and `0 × ∞` is NaN, so the gradient quietly becomes NaN.
Anomaly detection (slow, for debugging only) names the operation that produced it.

In [12]:
x = torch.tensor([0.0, 4.0], requires_grad=True)
(torch.sqrt(x) * torch.tensor([0.0, 1.0])).sum().backward()
print("gradient without anomaly detection:", x.grad.tolist())

x.grad = None
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                        # anomaly mode also prints the forward-pass traceback as a warning
    with torch.autograd.detect_anomaly():
        show_error(lambda: (torch.sqrt(x) * torch.tensor([0.0, 1.0])).sum().backward())

gradient without anomaly detection: [nan, 0.25]
RuntimeError: Function 'SqrtBackward0' returned nan values in its 0th output.


Without anomaly detection, the NaN just appears in the gradient and, one step later, in every weight. With it, PyTorch stops at `SqrtBackward0`,
the backward of the `sqrt` call. In practice, NaNs come from a learning rate that is too high, `log(0)` or division by zero in a custom loss,
float16 overflow, or bad input data; adding a small ε (`log(p + 1e-8)`) or using the built-in fused losses avoids most of them.

# 9. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Gradient clipping | none / by norm / by value | none: simplest, but one bad batch can ruin a run. By norm: caps the size, keeps the direction. By value: changes the direction | clip by norm at 1.0 for transformers and RNNs |
| Precision | float32 / bfloat16 autocast / float16 + GradScaler / FP8 | float32: simplest and slowest. bf16: about 2x faster and less memory, no scaler. fp16: same speed, needs loss scaling. FP8: faster again, only on the newest GPUs and with care | bf16 where supported, fp16 + scaler on older GPUs |
| Big batches | a bigger batch / gradient accumulation | bigger: fastest, needs memory. Accumulation: same update in less memory, more time per step; BatchNorm sees only the micro-batch | the biggest batch that fits, accumulation for the rest |
| Activation checkpointing | off / on | on: much less activation memory, about 20–30% slower training | on for large models or long sequences |
| Experiment tracking | log files / MLflow / W&B | files: no setup, hard to compare. MLflow: self-hosted, free, model registry. W&B: best dashboards, hosted | MLflow inside companies that self-host, W&B for teams that accept a hosted service |
| Multiple GPUs | DDP / FSDP or ZeRO / tensor and pipeline parallelism | DDP: simplest, fastest when the model fits on one GPU. FSDP: fits much larger models, more communication. Tensor and pipeline: needed for the largest models, complex | DDP; FSDP once the model and optimizer state don't fit |
| Debugging | train and hope / checks first | checks take minutes; a silent bug can waste days | the checklist in section 8 before any long run |

# 10. INTERVIEW QUESTIONS

### Q1. Your model doesn't fit in GPU memory. What can you do, in 30 seconds?
Cheapest first: mixed precision (bf16) halves activation memory; a smaller micro-batch with gradient accumulation keeps the same effective batch;
activation checkpointing trades compute for memory; 8-bit optimizers or Adafactor shrink the optimizer state; FSDP or ZeRO shard weights, gradients and optimizer state across GPUs;
for fine-tuning, LoRA or QLoRA trains only small adapters.

### Q2. What is gradient clipping, and when do you need it?
It caps the total gradient norm at a threshold by rescaling all gradients, keeping their direction. It protects against rare huge gradients that destabilize training,
common in RNNs and transformers (max_norm = 1.0). Logging the pre-clip norm also shows when training is going wrong.

### Q3. How does mixed precision training work?
Autocast runs matrix multiplies and convolutions in 16-bit floats and keeps numerically sensitive operations (softmax, norms, losses) in float32. Weights stay in float32,
so small updates aren't lost. It's faster on Tensor Cores and uses less activation memory.

### Q4. float16 vs bfloat16 for training? What is loss scaling?
float16 has more precision but a small range: small gradients underflow to zero and large values overflow. Loss scaling multiplies the loss before backward so gradients stay representable,
then unscales them in float32; GradScaler adjusts the scale dynamically and skips steps with inf/NaN. bfloat16 has float32's range, so it needs no scaling and is preferred where supported.

### Q5. How does gradient accumulation work, and what are its limits?
Run several micro-batches, divide each loss by their number, call backward on each without zeroing, then step once: the same gradient as one big batch.
It costs time, not memory. BatchNorm only sees micro-batch statistics, and the learning rate should match the effective batch size.

### Q6. What is activation checkpointing?
Instead of saving every intermediate activation for backward, save only some (for example each block's input) and recompute the rest during the backward pass.
It cuts activation memory substantially for about one extra forward pass, around 20–30% more compute.

### Q7. How does DDP work?
Each GPU runs a process with a full copy of the model. A DistributedSampler gives each a different shard of the data. After backward, DDP all-reduces (averages) the gradients across processes,
overlapping communication with computation, so every copy applies the same update and stays identical.

### Q8. DDP vs FSDP (or ZeRO)?
DDP replicates everything on every GPU, which is fast but limited by one GPU's memory. FSDP and ZeRO shard parameters, gradients and optimizer state across GPUs and gather each layer's weights only when needed,
so much larger models fit, at the cost of more communication.

### Q9. What are tensor and pipeline parallelism?
Tensor parallelism splits individual weight matrices across GPUs so one layer's matrix multiply runs on several GPUs at once. Pipeline parallelism puts different layers on different GPUs
and streams micro-batches through them. Very large LLMs combine both with data parallelism.

### Q10. Why call `sampler.set_epoch(epoch)` in distributed training?
DistributedSampler seeds its shuffle with the seed plus the epoch. Without `set_epoch`, every epoch uses the same order. Setting it changes the order every epoch while keeping all processes consistent.

### Q11. Why track experiments, and with what?
To compare runs reliably and reproduce the best one: parameters, metrics over time, code version and output files in one place. MLflow (self-hosted, model registry) or Weights & Biases (hosted dashboards).

### Q12. Your training loss isn't going down. How do you debug it?
Check the data (look at examples and labels after the DataLoader), the loss at initialization (about ln(classes)), and whether the model can overfit one small batch.
Then check gradient norms, try learning rates 10x up and down, compare with a simple baseline, and verify `train()`, `eval()` and `zero_grad()`.

### Q13. The loss became NaN. What do you do?
Find the first step where it happens; log the gradient norm and look for spikes. Common causes: a learning rate that is too high, missing warm-up, log of zero or division by zero in custom code,
float16 overflow, bad input values. Use anomaly detection to find the operation, add clipping, lower the learning rate, and use the built-in fused losses.

# 11. IN MY OWN WORDS

Write this from memory, without looking above:

- Gradient clipping, and where it goes in the training step:
- How mixed precision works, and when loss scaling is needed:
- How gradient accumulation gives the same update as a bigger batch:
- DDP vs FSDP:
- My first five checks when a model won't learn: